# Notebook to preprocess IFRC reports

In [1]:
%load_ext autoreload
%autoreload 2

## Steps
1. Load reports text from JSON (check it is the correct version, eventually redo scraping ourself)
2. Filter out unnessecary reports
3. Clean text
4. Separate sentences and tokenize
5. Add hazard category for each report (use Laura's reclassifying)
6. Add division according to header


In [23]:
import pandas as pd
import json
from collections import Counter
from src.text_processing_functions import *
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import copy as cp
import spacy
import regex as re
from src.data import *
from src.hazard_def import hazard_subtype_kw_searc
import chardet
#from spacy.language import Language  # For custom pipeline components
#from spacy_langdetect import LanguageDetector  # For language detection
import spacy_fastlang
import unicodedata

In [ ]:
file_path = DATA_IN_JSONS + '/filtered_report_types_nat_hazards_bugfix.json'

# Open and read the JSON file
with open(file_path, 'r') as json_file:
    all_ifrc_reports_info_unnested = json.load(json_file)

In [25]:
# Convert the JSON data into a Pandas DataFrame
data = pd.DataFrame(all_ifrc_reports_info_unnested)

## Filtering useless reports

In [ ]:
## filter out useless reports
filtered_reports = [
    disaster_report for disaster_report in all_ifrc_reports_info_unnested
    if disaster_report['appealType'] in ['Operations Update', 'DREF Operation', 'DREF Operation Final Report', 'DREF Operation Update']
]

#for tests
#filtered_reports = [
#    disaster_report for disaster_report in filtered_reports
#    if disaster_report["appealCode"] in ["MDRPK026", "MDRSV012"]
#]

##rename columns preprocessed by tais
for report in filtered_reports[:]:
    report["text_processed_orig"] = report["text_processed"]
    report["sentences_orig"] = report["sentences"]
    del report["text_processed"]
    del report["sentences"]

## Text preprocessing

In [27]:
#load libraries fo nlp
#not clear exactly which preprocessing steps must be undertaken
import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt_tab')
nltk.download('punkt')  # Download sentence tokenizer
nltk.download('stopwords') # Download stopwords

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lhasbini\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lhasbini\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lhasbini\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [28]:

#https://spacy.io/universe/project/spacy_fastlang
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("language_detector")

#or custom from https://medium.com/@j.boldsen.ryan/detecting-languages-with-spacy-and-spacy-langdetect-0b733a2a06d2
#from spacy.language import Language  # For custom pipeline components
#from spacy_langdetect import LanguageDetector  # For language detection
# Custom language detector factory function
#@Language.factory("language_detector")
#def create_language_detector(nlp, name):
#    return LanguageDetector() # Create the detector component

# Add language detector to the spaCy pipeline
#nlp.add_pipe("language_detector", last=True)

# Function to check if the text is in English
def detect_language(text):
    doc = nlp(text)
    return doc._.language  # Check if detected language is English

def rewrite_flood_affect(text):
    text = re.sub(r'\bood', "flood", text.lower())
    text = re.sub(r'\baect', "affect", text.lower())
    return text

def fix_pdf_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace('\x00', '')
    text= re.sub(r"\s+", " ", text.lower()).strip() #remove random whitespaces
    return text

In [29]:
test_text1 = "a\x00ected people by \x00ood afjjajkefa"
test_text2 = "aected people by ood afafs"
rewrite_flood_affect(test_text2)

'affected people by flood afafs'

In [30]:
#clean text and tokenize into sentences
format_numbers = False
std_units = False
not_eng = []
for item in filtered_reports[:]:
    if 'text' in item:
        #identify language and get rid of reports not in english
        item['language'] = detect_language(item['text'])
        if item['language'] != 'en':
            filtered_reports.remove(item)
            not_eng.append(item)
            continue
        #item['text_processed'] = fix_pdf_text(item['text_processed'])
        item['text_processed'] = clean_text(item['text'], format_numbers=format_numbers)
        #item['text_processed'] = rewrite_flood_affect(item['text_processed'])
        if std_units:
            item['text_processed'] = standardize_units(item['text_processed'])
        item['sentences'] = sent_tokenize(item['text_processed'])

    else: # drop reports without text
        filtered_reports.remove(item)

In [31]:
#for i in range(len(filtered_reports[0]["sentences"])):
#    if filtered_reports[0]["sentences"][i] != filtered_reports[0]["sentences_orig"][i]:
#        print('original: ' + filtered_reports[0]["sentences_orig"][i] + '\n\nmodified: '+filtered_reports[0]["sentences"][i])


In [32]:
from src.text_processing_functions import *
test = "\nThe dual crises of flooding and cholera outbreaks struck at a time when 24.8 million people in Sudan needed humanitarian assistance,\nover 10 million were internally displaced, and the country faced one of the worst food security crises in the world."
test = "24.8 million people affected in seven forks dam"
#test_df = data.where(data.appealCode.isin(['MDRSD034'])).dropna(how='all').iloc[0].text
clean_text(test, format_numbers=True)


'24800000.0 people affected in 7.0 forks dam'

In [33]:
# Add the ISO code to each dict in the list

import pycountry

for report in filtered_reports[:]:
    country_name = report.get("location")
    try:
        # Lookup the ISO code using pycountry
        country = pycountry.countries.get(name=country_name)
        if country:
            report["iso_code"] = country.alpha_3  # Adds the ISO 3166-1 Alpha-3 code
        else:
            report["iso_code"] = "Unknown"
    except KeyError:
        report["iso_code"] = "Unknown"


## Add natural hazard type and filter out other disasters

In [34]:
# Apply the function to the 'text_preprocessed' column of the DataFrame
for report in filtered_reports[:]:
    report['hazards_found_kw'] = check_hazard_type_keyword(report['text_processed'], hazard_subtype_kw_searc)

In [35]:
# filter out reports with no identified hazard with keyword searc but keep those where disasterType is related to a nat haz
filtered_reports_hazonly = cp.deepcopy(filtered_reports)
for report in filtered_reports_hazonly[:]:
    if (len(report['hazards_found_kw']) == 0):# and (report['disasterTypeReclassified'] not in disasterType_nathaz)):
        filtered_reports_hazonly.remove(report)

In [36]:
joined_df = pd.concat([pd.DataFrame(filtered_reports).groupby('disasterTypeReclassified').count()['reportName'], pd.DataFrame(filtered_reports_hazonly).groupby('disasterTypeReclassified').count()['reportName']],
                      axis=1, keys=["all", "hazonly"])

In [37]:
joined_df.sum()

all        5
hazonly    5
dtype: int64

In [38]:
# joined_df.plot(kind='bar', figsize=(10, 6))

## Select subsections containing natural hazard info

In [39]:
for report in filtered_reports_hazonly[:]:
    report['nathaz_text'] = select_hazard_description(report['sentences'], match_above=False)

In [40]:
#also for all haz
for report in filtered_reports[:]:
    report['nathaz_text'] = select_hazard_description(report['sentences'], match_above=False)

In [22]:
import importlib
import sys
importlib.reload(sys.modules['src.text_processing_functions'])
from src.text_processing_functions import select_hazard_description

In [23]:
data = pd.DataFrame(filtered_reports)
test_text1 = ['DREF Operation n° MDRSV012 GLIDE: n° TC-2018 -000167 -SLV Date of issue: 25 June 2019 Date of disaster: 15 October 2018 Operation start date: 1 December 2018 Operation end date: 15 February 2019 DREF allocated: 150,671 Swiss francs (CHF) Number of people affected: 7,085 (1,417 families ) Number of people assisted: 2,090 (418 families ) Host National Society presence (n° of volunteers, staff, branches): The Salvador ean Red Cross Society (SRCS) has one headquarter, 63 branches throughout the country, 2,239 volunteers and 275 staff.', '75 volunteers have been trained as National Intervention T eams (NITs) with different specialties (Water, Sanitation and Hygiene Promotion, Logistics, General, ZIKA and Vector Control, Psychosocial Support (PSS)) and 35 active volunteers trained in the Damage Assessment and Needs Analysis (DANA) assessment tool.', 'Red Cross Red Crescent Movement partners actively involved in the operation: International Federation of Red Cross and Red Crescent Societies (IFRC).', 'Other partner organizations actively involved in the operation : El Salvador Civil Protection System and departmental, municipal and community commissions , Ministry of Health, the Medical Emergency System (SEM for its acronym in Spanish), the Solidarity Fund for Health (FOSALUD for its acronym in Spanish), National Administration of Aqueducts and Sewers (ANDA ), Municipal Mayors’ Offices, Ministry of Education, Departmental, Municipal and Community Civil Protection Commissions, Health Committees in communities, and ADESCOS .', 'The major donors and partners of the Disaster Relief Emergency Fund (DREF) include the Red Cross Societies and governments of Belgium, Britain, Canada, Denmark, German, Ireland, Italy, Japan, Luxembourg, New Zealand, Norway, Republic of Korea, Spain, Sweden and Switzerland, as well as DG ECHO and Blizzard Entertainment, Mondelez International Foundation, and Fortive Corporation and other corporate and private donors.', 'The IFRC, on behalf of the national society, would like to extend thanks to all for their generous contributions.', 'ECHO and the government of Canada have replenished the DREF in the occasion of this operation.', 'The total amount spent under this DREF operation was 118,630 CHF.', 'The remaining balance of 32,041 CHF will be reimbursed to the Disaster Relief Emergency Fund .', '< For the Final Financial Report, click here .', 'For contact information, click here .', '> A.', 'Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', '14 located near the Honduran Atlantic coast.', 'On 7 October, the tropical depression was upgraded to Tropical Storm Michael, which continued moving north over the Yucatán channel toward the System declared a Green Alert for the entire country.', 'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalitie s in Morazán department and two in La Union department.', 'A Green Alert remained in place for the rest of the country .', 'The rains have affected the entire country.', 'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel; the cantons of San Felipe and Las Tunas in La Unión department; the cantons of Capitán Lazo and Puerto Parada in the municipality of Usulut án; as well as the canton of Metalío in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador: Floods Photo: Area affected by Hurricane Michael.', 'Source: SRCS 2019.', '2 | P a g e cantons of San Diego and San Rafael Abajo in the municipality of La Libertad in central El Salvador.', "These floods affected most municipalities located along the country's coast .", 'Report on the effects of the National Civil Protection System Event: Yellow and green warning for rain 11:30 AM 06/10/2018 Preliminary report accumulated no.', '3, affectations from 16:30 hours on 05/10/2018 to 06:00 hours on 09/10/2018.', 'Affected transit routes TOTAL Flooded highways 1 Flooded roads 3 Affected highways 24 Affected rods 31 Isolated communities 1 Affected people TOTAL Injured 14 Dead 4 Sheltered 1,090 Active shelters 13 Other TOTAL Fallen trees 42 Branches of fallen trees 5 Landslides 74 Floods 1 Overflowed rivers 7 Subsidences 1 Vehicles directly affected by the event 5 Affected homes and buildings TOTAL Affected homes 6 Flooded homes 1,409 Destroyed homes 2 Other buildings affected 1 Other buildings destroyed (collapsed walls) 6 Summary of response Overview of Host National Society The Salvador ean Red Cross Society (SRCS) constantly monitored the situation through its branches across the country since the onset of low-pressure system No .', "jeej", "jooj", "summary of the response :)"]
#test_text2 = data[data["appealCode"]=="MDRSV012"]["sentences"].iloc[0]
#test_text3 = data[data["appealCode"]=="MDRRW022"]["sentences"].iloc[0]
test_text4 = ["basdj", 'coordination and partnerships red cross red crescent move ment the representatives of each partnering organisation (vanuatu r ed cross, australian r ed cross, french r ed cross and ifrc) have a close working relationship and liaise frequently as they are all based in the same vanuatu r ed cross headquarter s. informat ion sharing is ongoing with red cross partners that are contributing to the appeal including partners that do not have a delegation in vanuatu such as new zealand r ed cross and american r ed cross.',
             "coordination and partnerships","adfnjadflam", "sdfnjakldfma"]
test_text = test_text4
print(test_text)
print("\n"*2)
select_hazard_description(test_text)

['basdj', 'coordination and partnerships red cross red crescent move ment the representatives of each partnering organisation (vanuatu r ed cross, australian r ed cross, french r ed cross and ifrc) have a close working relationship and liaise frequently as they are all based in the same vanuatu r ed cross headquarter s. informat ion sharing is ongoing with red cross partners that are contributing to the appeal including partners that do not have a delegation in vanuatu such as new zealand r ed cross and american r ed cross.', 'coordination and partnerships', 'adfnjadflam', 'sdfnjakldfma']





['basdj',
 'coordination and partnerships red cross red crescent move ment the representatives of each partnering organisation (vanuatu r ed cross, australian r ed cross, french r ed cross and ifrc) have a close working relationship and liaise frequently as they are all based in the same vanuatu r ed cross headquarter s. informat ion sharing is ongoing with red cross partners that are contributing to the appeal including partners that do not have a delegation in vanuatu such as new zealand r ed cross and american r ed cross.',
 'coordination and partnerships',
 'adfnjadflam',
 'sdfnjakldfma']

## Save data

In [ ]:
fname_nathaz = 'nathaz_ifrc_reports_info_processed_v030925'
if format_numbers:
    fname_nathaz = fname_nathaz + '_format_nb'
if std_units:
    fname_nathaz = fname_nathaz + '_std_units'
with open(DATA_IN_JSONS +fname_nathaz+'.json', 'w') as f:
    json.dump(filtered_reports_hazonly, f, indent=4)

In [ ]:
fname_all = 'all_ifrc_reports_info_processed_v030925'
if format_numbers:
    fname_all = fname_all + '_format_nb'
if std_units:
    fname_all = fname_all + '_std_units'
with open(DATA_IN_JSONS +fname_all+".json", 'w') as f:
    json.dump(filtered_reports, f, indent=4)